# Qwen 3.5 4B через llama.cpp

> Сборка llama.cpp и модель берутся из каталогов текущего ZEMI Instance через `zemi.env.path`.

## Запуск модели

In [11]:
from zemi import env, exp, toml

config = toml.load(env.path.comp / "runme.toml")
server = config["servers"][0]
model_config = server["models"][0]

download_config = {
    "llama_build": server["llama_build"],
    "model_source": model_config["source"],
    "model_owner": model_config["owner"],
    "model_repository": model_config["repository"],
    "model_filename": model_config["filename"],
}

exp.llama.download(**download_config)

llama.cpp b9222 уже скачан: C:\Users\Axoman\Documents\ZEMI\_llamas\llama--b9222
Модель уже скачана: C:\Users\Axoman\Documents\ZEMI\_models\hf--bartowski--Qwen_Qwen3.5-4B-GGUF--Qwen_Qwen3.5-4B-Q4_K_M\Qwen_Qwen3.5-4B-Q4_K_M.gguf


(WindowsPath('C:/Users/Axoman/Documents/ZEMI/_llamas/llama--b9222'),
 WindowsPath('C:/Users/Axoman/Documents/ZEMI/_models/hf--bartowski--Qwen_Qwen3.5-4B-GGUF--Qwen_Qwen3.5-4B-Q4_K_M/Qwen_Qwen3.5-4B-Q4_K_M.gguf'))

In [12]:
server_config = {
    **download_config,
    "alias": model_config["alias"],
    "ctx_size": model_config["ctx_size"],
    "threads": model_config["threads"],
    "threads_batch": model_config["threads_batch"],
    "reasoning": model_config["reasoning"],
    "startup_timeout": model_config["startup_timeout"],
}

exp.llama.restart(**server_config)
llama_url = exp.llama.SERVER_URL

Работающий llama-server не найден
llama-server запущен и готов, PID: 14144


## Диалог с моделью

In [14]:
import json

from guidance import assistant, json as gen_json, system, user
from guidance.models import OpenAI


model = OpenAI(
    model=model_config["alias"],
    base_url=f"{llama_url}/v1",
    api_key="not-needed",
    timeout=300.0,
    echo=False,
)

DATA_DIR = env.path.comp / "data/case01"

EXCEL_FILES = [
    DATA_DIR / "Отчет 1.xlsx",
    DATA_DIR / "Отчет 2.xlsx",
    DATA_DIR / "Отчет 3.xlsx",
]


excel_context = "\n\n".join(exp.excel.excel_to_text(path) for path in EXCEL_FILES)

print("Скачанные файлы:")
for path in EXCEL_FILES:
    print(f"- {path.name}: {path.stat().st_size} байт")

print(f"\nРазмер подготовленного текста: {len(excel_context)} символов")

TASK = """
Обработай все переданные Excel-отчёты.

Для каждого файла извлеки:
- название города/филиала;
- дату выгрузки;
- руководителя филиала;
- строки таблицы с полями date, article, cost.

Не включай строку «Итого».
Не выдумывай отсутствующие данные.
""".strip()

REPORTS_SCHEMA = {
    "type": "object",
    "properties": {
        "reports": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "source_file": {"type": "string"},
                    "city": {"type": "string"},
                    "export_date": {"type": "string", "pattern": "^[0-9]{4}-[0-9]{2}-[0-9]{2}$"},
                    "manager": {"type": "string"},
                    "transactions": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "date": {"type": "string", "pattern": "^[0-9]{4}-[0-9]{2}-[0-9]{2}$"},
                                "article": {"type": "string"},
                                "cost": {"type": "number"},
                            },
                            "required": ["date", "article", "cost"],
                            "additionalProperties": False,
                        },
                    },
                },
                "required": ["source_file", "city", "export_date", "manager", "transactions"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["reports"],
    "additionalProperties": False,
}

print("\nОтправляю три Excel-файла модели...")

lm = model
with system():
    lm += (
        "Ты аккуратно преобразуешь содержимое Excel-отчётов в строго "
        "структурированные данные. Не выдумывай отсутствующие значения."
    )
with user():
    lm += f"{TASK}\n\nДАННЫЕ EXCEL:\n{excel_context}"
with assistant():
    lm += gen_json(
        name="reports_json",
        schema=REPORTS_SCHEMA,
        temperature=0.0,
        max_tokens=2048,
    )

text = lm["reports_json"]
result = json.loads(text)

print("\nОтвет модели:\n")
print(json.dumps(result, ensure_ascii=False, indent=2))


Скачанные файлы:
- Отчет 1.xlsx: 10365 байт
- Отчет 2.xlsx: 10291 байт
- Отчет 3.xlsx: 10530 байт

Размер подготовленного текста: 962 символов

Отправляю три Excel-файла модели...


APIConnectionError: Connection error.

In [ ]:
result

{
  "reports": [
    {
      "source_file": "Отчет 1.xlsx",
      "city": "Москва",
      "export_date": "2026-04-30",
      "manager": "Иванов И.И.",
      "transactions": [
        {"date": "2026-03-14", "article": "15", "cost": 83204},
        {"date": "2026-03-24", "article": "61", "cost": 53307},
        {"date": "2026-03-09", "article": "56", "cost": 67750},
        {"date": "2026-05-17", "article": "65", "cost": 21224},
        {"date": "2026-04-21", "article": "78", "cost": 65206},
        {"date": "2026-03-27", "article": "61", "cost": 40616},
        {"date": "2026-02-14", "article": "14", "cost": 33626}
      ]
    },
    {
      "source_file": "Отчет 2.xlsx",
      "city": "Санкт-Петербург",
      "export_date": "2026-04-30",
      "manager": "Петров П.П.",
      "transactions": [
        {"date": "2026-01-27", "article": "21", "cost": 3515},
        {"date": "2026-05-11", "article": "48", "cost": 1969},
        {"date": "2026-02-18", "article": "33", "cost": 2964},
       

'{\n  "reports": [\n    {\n      "source_file": "Отчет 1.xlsx",\n      "city": "Москва",\n      "export_date": "2026-04-30",\n      "manager": "Иванов И.И.",\n      "transactions": [\n        {"date": "2026-03-14", "article": "15", "cost": 83204},\n        {"date": "2026-03-24", "article": "61", "cost": 53307},\n        {"date": "2026-03-09", "article": "56", "cost": 67750},\n        {"date": "2026-05-17", "article": "65", "cost": 21224},\n        {"date": "2026-04-21", "article": "78", "cost": 65206},\n        {"date": "2026-03-27", "article": "61", "cost": 40616},\n        {"date": "2026-02-14", "article": "14", "cost": 33626}\n      ]\n    },\n    {\n      "source_file": "Отчет 2.xlsx",\n      "city": "Санкт-Петербург",\n      "export_date": "2026-04-30",\n      "manager": "Петров П.П.",\n      "transactions": [\n        {"date": "2026-01-27", "article": "21", "cost": 3515},\n        {"date": "2026-05-11", "article": "48", "cost": 1969},\n        {"date": "2026-02-18", "article": "3

## Остановка модели

Выполни эту ячейку, когда модель больше не нужна.

In [ ]:
exp.llama.stop()

llama-server остановлен, PID: 15572


True